# Warsaw EV - Analysis & Visuals
Using `dataset/sessions_enriched.csv` (sessions + station/customer/district context)

In [9]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

FIGS_DIR = "figs"
os.makedirs(FIGS_DIR, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIGS_DIR, name), dpi=150, bbox_inches="tight")

In [10]:
enriched_path = "../dataset/sessions_enriched.csv"

se = pd.read_csv(enriched_path, parse_dates=["session_start_time"])

print("Shape:", se.shape)

Shape: (114000, 24)


In [11]:
import folium
import pandas as pd
from branca.element import Element

# --- choose a stations-like dataframe to plot ---
# If you have the enriched df `se`, build a unique stations view from it (avoids duplicates).
if 'se' in globals():
    stations_map = (
        se[['station_id','latitude','longitude','operator_name','plugs_count','station_income_tier']]
        .dropna(subset=['latitude','longitude'])
        .drop_duplicates('station_id')
    )

# --- color mapping (normalize variants to Low/Mid/High) ---
tier_colors = {
    'low': 'blue',
    'mid': 'green',
    'high': 'red'
}

def normalize_tier(t):
    t = str(t).strip()
    return {
        'Low-Mid':'Low',
        'Mid-Range':'Mid',
        'Medium':'Mid',
        'LOW':'Low',
        'MID':'Mid',
        'HIGH':'High'
    }.get(t, t if t in ['Low','Mid','High'] else 'Unknown')

# --- build the map ---
warsaw_center = [52.2297, 21.0122]
m = folium.Map(location=warsaw_center, zoom_start=12)

for _, row in stations_map.iterrows():
    lat, lon = row['latitude'], row['longitude']
    if pd.isna(lat) or pd.isna(lon):
        continue

    tier = row.get('station_income_tier', 'Unknown')
    color = tier_colors.get(tier, 'gray')

    popup = (
        f"<b>Station:</b> {row.get('station_id','?')}<br>"
        f"<b>Operator:</b> {row.get('operator_name','?')}<br>"
        f"<b>Plugs:</b> {row.get('plugs_count','?')}<br>"
        f"<b>Income Tier:</b> {tier}"
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=popup
    ).add_to(m)

# --- legend that matches the colors above ---
legend_html = """
<div style="
    position: fixed; bottom: 40px; left: 40px; width: 160px;
    border: 2px solid #888; z-index: 9999; font-size: 14px;
    background: white; padding: 10px; border-radius: 8px;">
    <b>Income Tier</b><br>
    <div style="margin-top:6px;">
      <span style="display:inline-block;width:12px;height:12px;background:blue;margin-right:8px;border:1px solid #555;"></span> Low<br>
      <span style="display:inline-block;width:12px;height:12px;background:green;margin-right:8px;border:1px solid #555;"></span> Mid<br>
      <span style="display:inline-block;width:12px;height:12px;background:red;margin-right:8px;border:1px solid #555;"></span> High<br>
      <span style="display:inline-block;width:12px;height:12px;background:gray;margin-right:8px;border:1px solid #555;"></span> Unknown
    </div>
</div>
"""
m.get_root().html.add_child(Element(legend_html))

m

In [12]:

se["station_income_tier"].unique()

array(['low', 'mid', 'high'], dtype=object)